In [1]:
import datetime
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification
from transformers import AutoTokenizer, TFAutoModel, TFAutoModelForSequenceClassification
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling1D
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

/Users/adityagoyal/SRH/Advanced Programming/fakenews/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# loading the preprocessed data
articledf = pd.read_csv('../data/article_preprocessed.csv')

In [ ]:
# Defining the features and labels
texts = articledf['tokens'].tolist()
labels = articledf['target'].values

In [ ]:
# Using DistilBERT as Transformer choice (Quick and efficient)
def create_distilbert_model():
    model_name = 'distilbert-base-uncased'
    tokenizer = DistilBertTokenizer.from_pretrained(model_name)
    
    max_len = 256
    
    def encode_texts_fast(texts, tokenizer, max_len):
        """Faster tokenization with batch processing"""
        encoded = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors='tf'
        )
        return encoded['input_ids'], encoded['attention_mask']
    
    print("Tokenizing with DistilBERT...")
    input_ids, attention_masks = encode_texts_fast(texts, tokenizer, max_len)
    
    # Use pre-trained classification model directly
    model = TFDistilBertForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1,
        output_attentions=False,
        output_hidden_states=False
    )
    
    model.compile(
        optimizer="adam",
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    
    return model, tokenizer, input_ids, attention_masks, max_len

In [ ]:
# defining model, tokenizer, attention masks and other values using distilbert
model, tokenizer, input_ids, attention_masks, max_len = create_distilbert_model()

Tokenizing with DistilBERT...


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_layer_norm.weight', 'vocab_transform.bias', 'vocab_projector.bias', 'vocab_transform.weight', 'vocab_layer_norm.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

In [ ]:
# converting attnetion masks and input ids to numpy because train test split doesn't accept tensors
def convert_to_numpy(tensor):
    """Safely convert tensor to numpy array"""
    if hasattr(tensor, 'numpy'):
        return tensor.numpy()
    elif isinstance(tensor, tf.Tensor):
        return tensor.numpy()
    else:
        return np.array(tensor)

input_ids_np = convert_to_numpy(input_ids)
attention_masks_np = convert_to_numpy(attention_masks)

print(f"Input IDs shape: {input_ids_np.shape}")
print(f"Attention masks shape: {attention_masks_np.shape}")
print(f"Labels shape: {np.array(labels).shape}")

Input IDs shape: (44898, 256)
Attention masks shape: (44898, 256)
Labels shape: (44898,)


In [10]:
# Split the data
X_train_ids, X_test_ids, X_train_masks, X_test_masks, y_train, y_test = train_test_split(
    input_ids_np, attention_masks_np, labels, test_size=0.25, random_state=42
)
print(f"Training set size: {len(X_train_ids)}")
print(f"Test set size: {len(X_test_ids)}")
print(f"Max sequence length: {max_len}")


Training set size: 33673
Test set size: 11225
Max sequence length: 256


In [ ]:
# Use larger batch size for better GPU utilization
batch_size = 32
epochs = 3

# Mixed precision for faster training (if supported)
try:
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print("Mixed precision enabled for faster training")
except:
    print("Mixed precision not available, using float32")


Mixed precision enabled for faster training


In [13]:
# Training
start_time = datetime.datetime.now()

history = model.fit(
    [X_train_ids, X_train_masks], y_train,
    validation_split=0.1,
    epochs=epochs,
    batch_size=batch_size,
    verbose=1
)

end_time = datetime.datetime.now()
training_time = end_time - start_time
print(f"\nTraining completed in: {training_time}")


Epoch 1/3
  9/948 [..............................] - ETA: 1:41:37 - loss: 0.7201 - accuracy: 0.4375

KeyboardInterrupt: 

In [ ]:
# Plot training history
plt.figure(figsize=(12, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Training & Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Evaluate the model
print("\nEvaluating model...")
test_loss, test_accuracy = model.evaluate([X_test_ids, X_test_masks], y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Make predictions
y_pred_logits = model.predict([X_test_ids, X_test_masks], batch_size=64)
y_pred_probs = tf.nn.sigmoid(y_pred_logits.logits)
y_pred = (y_pred_probs > 0.5).numpy().astype(int)
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'],
            yticklabels=['Real', 'Fake'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))


In [ ]:
# Save the model
print("\nSaving model...")
model.save_pretrained("../models/efficient_transformer_fake_news")
tokenizer.save_pretrained("../models/efficient_transformer_tokenizer")

print(f"\nModel training completed successfully!")
print(f"Total training time: {training_time}")
print(f"Final test accuracy: {test_accuracy:.4f}")